<a href="https://colab.research.google.com/github/TegarReskiPratama/Analisis-Perbandingan-Kinerja-Model-Transformer-IndoBERT-IndoBERTweet-dan-XLM-RoBERTa-/blob/main/Analisis_Perbandingan_Kinerja_Model_Transformer_IndoBERT%2C_IndoBERTweet%2C_dan_XLM_RoBERTa_dalam_Deteksi_Promosi_Judi_Online_Berbahasa_Indonesia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers datasets accelerate evaluate
!pip install -q sentencepiece

In [ ]:
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df1 = pd.read_csv("youtube_chat_jogja_clean.csv")
df2 = pd.read_csv("data_komentar_dengan_prediksi - data_komentar_dengan_prediksi(2).csv")

print(df1.shape)
print(df2.shape)

# Ambil Kolom yang Digunakan

In [ ]:
df1 = df1[['cleaned_message','label']]
df1.columns = ['text','label']

df2 = df2[['komentar_clean','label']]
df2.columns = ['text','label']

print(df1.head())
print(df2.head())

In [ ]:
df = pd.concat([df1,df2], ignore_index=True)

print("Jumlah Data:", len(df))
df.head()

# Cleaning

In [ ]:
df.dropna(inplace=True)

df['text'] = df['text'].astype(str)

df = df[df['text'].str.len() > 3]

df.drop_duplicates(inplace=True)

df.reset_index(drop=True, inplace=True)

print(df.shape)

# WordCloud Judol

In [ ]:
from wordcloud import WordCloud
judol_text = " ".join(
    df[df['label']==1]['text'].astype(str)
)

wc = WordCloud(
    width=1000,
    height=500
).generate(judol_text)

plt.figure(figsize=(12,6))
plt.imshow(wc)
plt.axis("off")
plt.show()

# WordCloud Non Judol

In [ ]:
normal_text = " ".join(
    df[df['label']==0]['text'].astype(str)
)

wc = WordCloud(
    width=1000,
    height=500
).generate(normal_text)

plt.figure(figsize=(12,6))
plt.imshow(wc)
plt.axis("off")
plt.show()

# Cleaning Function

In [ ]:
def clean_text(text):

    text = str(text)

    text = re.sub(r"http\S+"," ",text)

    text = re.sub(r"@\w+"," ",text)

    text = re.sub(r"#\w+"," ",text)

    text = re.sub(r"[^a-zA-Z\s]"," ",text)

    text = text.lower()

    text = re.sub(r"\s+"," ",text)

    return text.strip()

In [ ]:
import re
import string
df['text'] = df['text'].apply(clean_text)

# Distribusi Label

In [ ]:
df['label'].value_counts()

# Visualisasi Label


In [ ]:
plt.figure(figsize=(5,4))

sns.countplot(
    x=df['label']
)

plt.title("Distribusi Label")
plt.show()

# Balancing Dataset

In [ ]:
from sklearn.utils import resample

df_major = df[df['label']==0]
df_minor = df[df['label']==1]

df_major_down = resample(
    df_major,
    replace=False,
    n_samples=len(df_minor),
    random_state=42
)

df_balanced = pd.concat([
    df_major_down,
    df_minor
])

df_balanced = df_balanced.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

df_balanced['label'].value_counts()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.countplot(
    x='label',
    data=df_balanced
)

plt.title("Distribusi Setelah Balancing")
plt.show()

# Train-Test Split

 Setelah balancing:

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df_balanced,
    test_size=0.2,
    stratify=df_balanced['label'],
    random_state=42
)

# Evaluasi

In [ ]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, predictions)

    prec = precision_score(labels, predictions)

    rec = recall_score(labels, predictions)

    f1 = f1_score(labels, predictions)

    return {
        "accuracy":acc,
        "precision":prec,
        "recall":rec,
        "f1":f1
    }

# List Model

In [ ]:
models = {
    "IndoBERT":"indobenchmark/indobert-base-p1",
    "IndoBERTweet":"indolem/indobertweet-base-uncased",
    "XLM-RoBERTa":"xlm-roberta-base"
}

# Variabel Hasil

In [ ]:
results = []
confusion_matrices = {}

# MODEL 1 : INDOBERT

In [ ]:
MODEL_NAME = "indobenchmark/indobert-base-p1"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

In [ ]:
def tokenize_function(examples):

    return tokenizer(
        examples["text"],
        truncation=True,
        padding=False,
        max_length=64
    )

# Convert pandas DataFrames to Hugging Face Dataset objects
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

train_tokenized = train_dataset.map(
    tokenize_function,
    batched=True
)

test_tokenized = test_dataset.map(
    tokenize_function,
    batched=True
)

In [ ]:
train_tokenized = train_tokenized.remove_columns(
    [col for col in train_tokenized.column_names
     if col not in ['input_ids','attention_mask','label']]
)

test_tokenized = test_tokenized.remove_columns(
    [col for col in test_tokenized.column_names
     if col not in ['input_ids','attention_mask','label']]
)

In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    ignore_mismatched_sizes=True
)

In [ ]:
training_args = TrainingArguments(
    output_dir="./indobert",

    num_train_epochs=1,

    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,

    eval_strategy="epoch",

    save_strategy="no",

    logging_steps=50,

    fp16=torch.cuda.is_available(),

    report_to="none"
)

In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

In [ ]:
pred = trainer.predict(test_tokenized)

y_pred = np.argmax(
    pred.predictions,
    axis=1
)

y_true = test_df["label"].values

acc_indobert = accuracy_score(y_true,y_pred)
prec_indobert = precision_score(y_true,y_pred)
rec_indobert = recall_score(y_true,y_pred)
f1_indobert = f1_score(y_true,y_pred)

print(classification_report(
    y_true,
    y_pred
))

# MODEL 2 : INDOBERTWEET

In [ ]:
MODEL_NAME = "indolem/indobertweet-base-uncased"

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "indolem/indobertweet-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True
)

In [ ]:
from sklearn.utils import resample

df0 = df_balanced[df_balanced["label"] == 0]
df1 = df_balanced[df_balanced["label"] == 1]

df0 = resample(
    df0,
    replace=False,
    n_samples=200,
    random_state=42
)

df1 = resample(
    df1,
    replace=False,
    n_samples=200,
    random_state=42
)

df_small = pd.concat([df0, df1])

df_small = df_small.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print(df_small["label"].value_counts())

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df_small,
    test_size=0.2,
    stratify=df_small["label"],
    random_state=42
)

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(
    train_df.reset_index(drop=True)
)

test_dataset = Dataset.from_pandas(
    test_df.reset_index(drop=True)
)

In [ ]:
def tokenize_function(examples):

    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=32
    )

train_tokenized = train_dataset.map(
    tokenize_function,
    batched=True
)

test_tokenized = test_dataset.map(
    tokenize_function,
    batched=True
)

In [ ]:
train_tokenized = train_tokenized.remove_columns(
    [col for col in train_tokenized.column_names
     if col not in ["input_ids","attention_mask","label"]]
)

test_tokenized = test_tokenized.remove_columns(
    [col for col in test_tokenized.column_names
     if col not in ["input_ids","attention_mask","label"]]
)

In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    ignore_mismatched_sizes=True
)

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./indobertweet",

    num_train_epochs=1,

    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,

    eval_strategy="no",

    save_strategy="no",

    logging_steps=10,

    report_to="none"
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    data_collator=data_collator
)

In [ ]:
trainer.train()

In [ ]:
predictions = trainer.predict(
    test_tokenized
)

y_pred = np.argmax(
    predictions.predictions,
    axis=1
)

y_true = test_df["label"].values

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred)
rec = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print("Accuracy :", acc)
print("Precision:", prec)
print("Recall   :", rec)
print("F1 Score :", f1)

print("\nClassification Report\n")
print(classification_report(y_true, y_pred))

# MODEL 3 : XLM-ROBERTA

In [ ]:
MODEL_NAME = "xlm-roberta-base"

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True
)

In [ ]:
print(train_df.shape)
print(test_df.shape)

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(
    train_df.reset_index(drop=True)
)

test_dataset = Dataset.from_pandas(
    test_df.reset_index(drop=True)
)

In [ ]:
def tokenize_function(examples):

    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=32
    )

train_tokenized = train_dataset.map(
    tokenize_function,
    batched=True
)

test_tokenized = test_dataset.map(
    tokenize_function,
    batched=True
)

In [ ]:
train_tokenized = train_tokenized.remove_columns(
    [col for col in train_tokenized.column_names
     if col not in ["input_ids","attention_mask","label"]]
)

test_tokenized = test_tokenized.remove_columns(
    [col for col in test_tokenized.column_names
     if col not in ["input_ids","attention_mask","label"]]
)

In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    ignore_mismatched_sizes=True
)

In [ ]:
training_args = TrainingArguments(
    output_dir="./xlmr",

    num_train_epochs=1,

    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,

    eval_strategy="no",

    save_strategy="no",

    logging_steps=10,

    fp16=True,

    report_to="none"
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,


    data_collator=data_collator
)

In [ ]:
trainer.train()

In [ ]:
predictions = trainer.predict(
    test_tokenized
)

y_pred = np.argmax(
    predictions.predictions,
    axis=1
)

y_true = test_df["label"].values

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

acc_xlmr = accuracy_score(y_true,y_pred)

prec_xlmr = precision_score(y_true,y_pred)

rec_xlmr = recall_score(y_true,y_pred)

f1_xlmr = f1_score(y_true,y_pred)

print("Accuracy :", acc_xlmr)
print("Precision:", prec_xlmr)
print("Recall   :", rec_xlmr)
print("F1 Score :", f1_xlmr)

print("\nClassification Report\n")

print(
    classification_report(
        y_true,
        y_pred
    )
)

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(
    y_true,
    y_pred
)

plt.figure(figsize=(5,4))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.title("Confusion Matrix XLM-RoBERTa")

plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

# Tabel Hasil Akhir

In [ ]:
results_df = pd.DataFrame({
    "Model":[
        "IndoBERT",
        "IndoBERTweet",
        "XLM-RoBERTa"
    ],
    "Accuracy":[
        acc_indobert,
        acc,
        acc_xlmr
    ],
    "Precision":[
        prec_indobert,
        prec,
        prec_xlmr
    ],
    "Recall":[
        rec_indobert,
        rec,
        rec_xlmr
    ],
    "F1":[
        f1_indobert,
        f1,
        f1_xlmr
    ]
})

results_df.sort_values(
    by="F1",
    ascending=False
)

# Confusion Matrix IndoBERT

In [ ]:
y_true
y_pred

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm_indobert = confusion_matrix(
    y_true,
    y_pred
)

plt.figure(figsize=(5,4))

sns.heatmap(
    cm_indobert,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.title("Confusion Matrix IndoBERT")

plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

# Confusion Matrix IndoBERTweet

In [ ]:
predictions = trainer.predict(test_tokenized)

y_pred_indobertweet = np.argmax(
    predictions.predictions,
    axis=1
)

y_true_indobertweet = test_df["label"].values

In [ ]:
cm_indobertweet = confusion_matrix(
    y_true_indobertweet,
    y_pred_indobertweet
)

plt.figure(figsize=(5,4))

sns.heatmap(
    cm_indobertweet,
    annot=True,
    fmt="d",
    cmap="Greens"
)

plt.title("Confusion Matrix IndoBERTweet")

plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

# Confusion Matrix XLM-RoBERTa

In [ ]:
predictions = trainer.predict(test_tokenized)

y_pred_xlmr = np.argmax(
    predictions.predictions,
    axis=1
)

y_true_xlmr = test_df["label"].values

In [ ]:
cm_xlmr = confusion_matrix(
    y_true_xlmr,
    y_pred_xlmr
)

plt.figure(figsize=(5,4))

sns.heatmap(
    cm_xlmr,
    annot=True,
    fmt="d",
    cmap="Oranges"
)

plt.title("Confusion Matrix XLM-RoBERTa")

plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

In [ ]:
print(y_true.shape)
print(y_pred.shape)

# Tabel Perbandingan Akhir

In [ ]:
import pandas as pd

results_df = pd.DataFrame({

    "Model":[
        "IndoBERT",
        "IndoBERTweet",
        "XLM-RoBERTa"
    ],

    "Accuracy":[
        acc_indobert,
        acc,
        acc_xlmr
    ],

    "Precision":[
        prec_indobert,
        prec,
        prec_xlmr
    ],

    "Recall":[
        rec_indobert,
        rec,
        rec_xlmr
    ],

    "F1-Score":[
        f1_indobert,
        f1,
        f1_xlmr
    ]
})

results_df

# Urutkan Berdasarkan F1-Score

In [ ]:
results_df = results_df.sort_values(
    by="F1-Score",
    ascending=False
)

results_df.reset_index(
    drop=True,
    inplace=True
)

results_df

#Bulatkan 4 Digit

In [ ]:
results_df.iloc[:,1:] = results_df.iloc[:,1:].round(4)

results_df

# Grafik Perbandingan

In [ ]:
import matplotlib.pyplot as plt

results_df.set_index("Model").plot(
    kind="bar",
    figsize=(10,6)
)

plt.title(
    "Perbandingan Kinerja Model Transformer"
)

plt.ylabel("Score")

plt.ylim(0,1)

plt.xticks(rotation=0)

plt.grid(True)

plt.show()

# Grafik F1 Score



In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))

plt.bar(
    results_df["Model"],
    results_df["F1-Score"]
)

plt.title(
    "Perbandingan F1-Score"
)

plt.ylabel("F1 Score")

plt.ylim(0,1)

for i,v in enumerate(results_df["F1-Score"]):
    plt.text(
        i,
        v+0.01,
        str(round(v,4)),
        ha="center"
    )

plt.show()

# Model Terbaik

In [ ]:
best_model = results_df.iloc[0]

print("="*50)

print("MODEL TERBAIK")

print("="*50)

print("Nama Model :", best_model["Model"])

print("Accuracy   :", best_model["Accuracy"])

print("Precision  :", best_model["Precision"])

print("Recall     :", best_model["Recall"])

print("F1-Score   :", best_model["F1-Score"])

# Ranking Model

In [ ]:
print("RANKING MODEL")

for i,row in results_df.iterrows():

    print(
        f"{i+1}. {row['Model']} | F1 = {row['F1-Score']:.4f}"
    )

# Tabel

In [ ]:
presentasi_df = results_df.copy()

for col in [
    "Accuracy",
    "Precision",
    "Recall",
    "F1-Score"
]:
    presentasi_df[col] = (
        presentasi_df[col] * 100
    ).round(2)

presentasi_df

# HASIL AKHIR PENELITIAN
# Analisis Perbandingan Kinerja Model Transformer
# IndoBERT, IndoBERTweet, dan XLM-RoBERTa

In [ ]:
best_model = results_df.iloc[0]

print("="*80)
print("HASIL AKHIR PENELITIAN")
print("="*80)

print("\nPERBANDINGAN KINERJA MODEL")
print(results_df)

print("\n")

print("="*80)
print("MODEL TERBAIK")
print("="*80)

print(f"Nama Model : {best_model['Model']}")
print(f"Accuracy   : {best_model['Accuracy']:.4f}")
print(f"Precision  : {best_model['Precision']:.4f}")
print(f"Recall     : {best_model['Recall']:.4f}")
print(f"F1-Score   : {best_model['F1-Score']:.4f}")

print("\n")

print("="*80)
print("ANALISIS HASIL")
print("="*80)

if best_model["Model"] == "IndoBERT":

    alasan = """
Model IndoBERT menjadi model terbaik karena dilatih secara khusus
menggunakan korpus Bahasa Indonesia sehingga mampu memahami
struktur kalimat, tata bahasa, dan konteks lokal dengan lebih baik.
Kemampuan ini membuat IndoBERT lebih efektif dalam membedakan
komentar promosi judi online dan komentar normal.
"""

elif best_model["Model"] == "IndoBERTweet":

    alasan = """
Model IndoBERTweet menjadi model terbaik karena dilatih pada data
media sosial berbahasa Indonesia. Karakteristik komentar judi online
yang cenderung menggunakan bahasa informal, singkatan, emoji,
dan pola komunikasi media sosial dapat dipahami lebih baik oleh
IndoBERTweet dibanding model lainnya.
"""

else:

    alasan = """
Model XLM-RoBERTa menjadi model terbaik karena memiliki kemampuan
multibahasa dan dilatih pada korpus yang sangat besar dari berbagai
bahasa. Hal ini membuat model mampu menghasilkan representasi teks
yang kuat sehingga dapat mengenali pola promosi judi online dengan
lebih baik.
"""

print(alasan)

print("\n")

print("="*80)
print("KESIMPULAN")
print("="*80)

print(f"""
Penelitian ini melakukan analisis perbandingan tiga model Transformer,
yaitu IndoBERT, IndoBERTweet, dan XLM-RoBERTa untuk mendeteksi
promosi judi online berbahasa Indonesia.

Berdasarkan hasil evaluasi menggunakan metrik Accuracy,
Precision, Recall, dan F1-Score, diperoleh bahwa model
{best_model['Model']} memberikan performa terbaik dengan
nilai F1-Score sebesar {best_model['F1-Score']:.4f}.

Hasil penelitian menunjukkan bahwa model Transformer mampu
digunakan untuk melakukan klasifikasi teks promosi judi online
secara efektif. Model {best_model['Model']} direkomendasikan
untuk digunakan pada sistem deteksi otomatis promosi judi online
karena memberikan performa paling tinggi dibandingkan model lain
yang diuji pada penelitian ini.
""")

print("\n")

print("="*80)
print("CONTOH HASIL PREDIKSI")
print("="*80)

sample_pred = pd.DataFrame({
    "Komentar": test_df["text"].values[:10],
    "Label Asli": y_true[:10],
    "Prediksi": y_pred[:10]
})

sample_pred["Label Asli"] = sample_pred["Label Asli"].map({
    0: "Non Judi Online",
    1: "Promosi Judi Online"
})

sample_pred["Prediksi"] = sample_pred["Prediksi"].map({
    0: "Non Judi Online",
    1: "Promosi Judi Online"
})

display(sample_pred)

print("\n")

print("="*80)
print("RANKING MODEL")
print("="*80)

for i, row in results_df.iterrows():

    print(
        f"{i+1}. {row['Model']} | "
        f"Accuracy={row['Accuracy']:.4f} | "
        f"Precision={row['Precision']:.4f} | "
        f"Recall={row['Recall']:.4f} | "
        f"F1={row['F1-Score']:.4f}"
    )